In [5]:
import requests
from pathlib import Path

DATASET_ID = "6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5"
CKAN_API = f"https://datos.produccion.gob.ar/api/3/action/package_show?id={DATASET_ID}"

# Consultar la API para obtener URLs actuales
response = requests.get(CKAN_API, timeout=30)
response.raise_for_status()
resources = response.json()["result"]["resources"]

# Construir mapa dia -> url desde los recursos que sean ZIPs de precios
url_por_dia = {
    r["name"].lower(): r["url"]
    for r in resources
    if r["url"].endswith(".zip") and "sepa_" in r["url"]
}

print("URLs disponibles:")
for dia, url in url_por_dia.items():
    print(f"  {dia}: {url}")

URLs disponibles:
  viernes: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/91bc072a-4726-44a1-85ec-4a8467aad27e/download/sepa_viernes.zip
  jueves: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/d076720f-a7f0-4af8-b1d6-1b99d5a90c14/download/sepa_jueves.zip
  miércoles: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/1e92cd42-4f94-4071-a165-62c4cb2ce23c/download/sepa_miercoles.zip
  martes: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/9dc06241-cc83-44f4-8e25-c9b1636b8bc8/download/sepa_martes.zip
  lunes: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/0a9069a9-06e8-4f98-874d-da5578693290/download/sepa_lunes.zip
  domingo: https://datos.produccion.gob.ar/dataset/6f47ec76-d1ce-4e34-a7e1-621fe9b1d0b5/resource/f8e75128-515a-436e-bf8d-5c63a62f2005/download/sepa_domingo.zip
  sábado: https://datos.pr

In [6]:
from datetime import date

DIAS_ES = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

dia_hoy = DIAS_ES[date.today().weekday()]
url = url_por_dia[dia_hoy]
dest = Path(f"../data/{dia_hoy}.zip")
dest.parent.mkdir(parents=True, exist_ok=True)

print(f"Descargando ZIP de {dia_hoy}...")
r = requests.get(url, stream=True, timeout=300)
r.raise_for_status()

dest.write_bytes(r.content)
print(f"Descargado: {dest} ({dest.stat().st_size / 1024 / 1024:.1f} MB)")

Descargando ZIP de martes...
Descargado: ../data/martes.zip (293.4 MB)


In [7]:
import zipfile

extract_dir = Path(f"../data/{dia_hoy}")
extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(dest) as z:
    print("Contenido del ZIP:")
    for name in z.namelist():
        info = z.getinfo(name)
        print(f"  {name} ({info.file_size / 1024 / 1024:.1f} MB)")
    z.extractall(extract_dir)

print(f"\nExtraído en: {extract_dir}")

Contenido del ZIP:
  2026-03-17/ (0.0 MB)
  2026-03-17/sepa_1_comercio-sepa-23_2026-03-17_09-05-11.zip (0.0 MB)
  2026-03-17/sepa_1_comercio-sepa-4_2026-03-17_09-05-11.zip (0.0 MB)
  2026-03-17/sepa_1_comercio-sepa-15_2026-03-17_09-05-11.zip (75.5 MB)
  2026-03-17/sepa_2_comercio-sepa-36_2026-03-17_01-05-08.zip (0.0 MB)
  2026-03-17/sepa_2_comercio-sepa-6_2026-03-17_01-05-08.zip (0.5 MB)
  2026-03-17/sepa_1_comercio-sepa-21_2026-03-17_09-05-11.zip (2.2 MB)
  2026-03-17/sepa_2_comercio-sepa-13_2026-03-17_01-05-08.zip (15.5 MB)
  2026-03-17/sepa_1_comercio-sepa-12_2026-03-17_09-05-11.zip (7.9 MB)
  2026-03-17/sepa_1_comercio-sepa-20_2026-03-17_09-05-11.zip (0.1 MB)
  2026-03-17/sepa_1_comercio-sepa-19_2026-03-17_09-05-11.zip (0.2 MB)
  2026-03-17/sepa_2_comercio-sepa-2_2026-03-17_01-05-08.zip (18.8 MB)
  2026-03-17/sepa_2_comercio-sepa-10_2026-03-17_01-05-08.zip (124.0 MB)
  2026-03-17/sepa_2_comercio-sepa-5_2026-03-17_01-05-08.zip (0.1 MB)
  2026-03-17/sepa_1_comercio-sepa-47_2026-03-17

In [8]:
import zipfile
from pathlib import Path

# Buscar ZIPs dentro de la carpeta extraída
inner_zips = list(extract_dir.rglob("*.zip"))
print(f"ZIPs encontrados: {len(inner_zips)}")
for z in inner_zips[:5]:
    print(f"  {z.name}")

# Extraer el primero y ver su contenido
sample_zip = inner_zips[0]
sample_dir = extract_dir / sample_zip.stem

with zipfile.ZipFile(sample_zip) as z:
    print(f"\nContenido de {sample_zip.name}:")
    for name in z.namelist():
        info = z.getinfo(name)
        print(f"  {name} ({info.file_size / 1024 / 1024:.1f} MB)")
    z.extractall(sample_dir)

print(f"\nExtraído en: {sample_dir}")

ZIPs encontrados: 18
  sepa_1_comercio-sepa-8_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-15_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-12_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-3_2026-03-17_09-05-11.zip
  sepa_1_comercio-sepa-19_2026-03-17_09-05-11.zip

Contenido de sepa_1_comercio-sepa-8_2026-03-17_09-05-11.zip:
  comercio.csv (0.0 MB)
  productos.csv (4.9 MB)
  sucursales.csv (0.0 MB)

Extraído en: ../data/martes/sepa_1_comercio-sepa-8_2026-03-17_09-05-11


In [12]:
import pandas as pd

df_comercio   = pd.read_csv(sample_dir / "comercio.csv",   sep="|", encoding_errors="replace")
df_productos  = pd.read_csv(sample_dir / "productos.csv",  sep="|", encoding_errors="replace")
df_sucursales = pd.read_csv(sample_dir / "sucursales.csv", sep="|", encoding_errors="replace")

# print("=== comercio.csv ===")
# print(df_comercio.shape)
# display(df_comercio.head(3))

print("=== productos.csv ===")
print(df_productos.shape)
display(df_productos.head(100))

# print("=== sucursales.csv ===")
# print(df_sucursales.shape)
# display(df_sucursales.head(3))

=== productos.csv ===
(43273, 17)


/tmp/ipykernel_20264/206441945.py:4: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_productos  = pd.read_csv(sample_dir / "productos.csv",  sep="|", encoding_errors="replace")


,id_comercio,id_bandera,id_sucursal,id_producto,productos_ean,productos_descripcion,productos_cantidad_presentacion,productos_unidad_medida_presentacion,productos_marca,productos_precio_lista,productos_precio_referencia,productos_cantidad_referencia,productos_unidad_medida_referencia,productos_precio_unitario_promo1,productos_leyenda_promo1,productos_precio_unitario_promo2,productos_leyenda_promo2
0,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO DOBLE TRAMA 40x45,1.0,uni.,10 y 11 S.A,999.0,999.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
1,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO CLASICA 42x40cm,1.0,uni.,10 y 11 S.A,730.0,730.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
2,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA AMERICANA 42x38cm.,1.0,uni.,10 y 11 S.A,939.0,939.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
3,8,1.0,1.0,7.798118e+12,1.0,FRANELA LEYENDA CLASICA COLOR 40x40cm,1.0,uni.,10 y 11 S.A,1490.0,1490.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
4,8,1.0,1.0,7.798118e+12,1.0,TRAPO PISO LEYENDA GRIS.PREM.FRIZ. 50x60,1.0,uni.,10 y 11 S.A,2390.0,2390.00,1.00,Precio x Unidad,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,8,1.0,1.0,7.790046e+12,1.0,GALLETITAS GRANIX SALVADO SIN SAL 230 GR,230.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),1269.0,5517.39,0.23,Precio x 1 Kg,NaN,NaN,NaN,NaN
96,8,1.0,1.0,7.790046e+12,1.0,GALLETITAS GRANIX SALVADO 0 COLEST. 240,240.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),1379.0,574.58,2.40,Precio x 100 Gr,NaN,NaN,NaN,NaN
97,8,1.0,1.0,7.790045e+12,1.0,GALLETITAS GRANAGUA SANDWICH 200gr,200.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),1169.0,584.50,2.00,Precio x 100 Gr,NaN,NaN,NaN,NaN
98,8,1.0,1.0,7.790045e+12,1.0,GALLETITAS GRANIX LIGHT SIN SAL 360 GRS.,360.0,Gr,ASOC.ARG.ADV.7mo DIA (a.granix),2109.0,5858.33,0.36,Precio x 1 Kg,NaN,NaN,NaN,NaN


In [13]:
df_productos.dtypes

id_comercio                              object
id_bandera                              float64
id_sucursal                             float64
id_producto                             float64
productos_ean                           float64
productos_descripcion                    object
productos_cantidad_presentacion         float64
productos_unidad_medida_presentacion     object
productos_marca                          object
productos_precio_lista                  float64
productos_precio_referencia             float64
productos_cantidad_referencia           float64
productos_unidad_medida_referencia       object
productos_precio_unitario_promo1        float64
productos_leyenda_promo1                float64
productos_precio_unitario_promo2        float64
productos_leyenda_promo2                float64
dtype: object

In [29]:
print(df_productos.shape)
print(df_productos.dropna().shape)

selected_columns = ["id_comercio", "id_bandera", "id_sucursal", "id_producto", "productos_ean", "productos_descripcion", "productos_cantidad_presentacion", "productos_unidad_medida_presentacion", "productos_marca", "productos_precio_lista", "productos_precio_referencia", "productos_cantidad_referencia", "productos_unidad_medida_referencia"]
print(df_productos[selected_columns].dropna().shape)

filtered_df = df_productos[selected_columns].dropna()
filtered_df.head(5)

(43273, 17)
(0, 17)
(43110, 13)


,id_comercio,id_bandera,id_sucursal,id_producto,productos_ean,productos_descripcion,productos_cantidad_presentacion,productos_unidad_medida_presentacion,productos_marca,productos_precio_lista,productos_precio_referencia,productos_cantidad_referencia,productos_unidad_medida_referencia
0,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO DOBLE TRAMA 40x45,1.0,uni.,10 y 11 S.A,999.0,999.0,1.0,Precio x Unidad
1,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA PABILO CLASICA 42x40cm,1.0,uni.,10 y 11 S.A,730.0,730.0,1.0,Precio x Unidad
2,8,1.0,1.0,7.798118e+12,1.0,REJILLA LEYENDA AMERICANA 42x38cm.,1.0,uni.,10 y 11 S.A,939.0,939.0,1.0,Precio x Unidad
3,8,1.0,1.0,7.798118e+12,1.0,FRANELA LEYENDA CLASICA COLOR 40x40cm,1.0,uni.,10 y 11 S.A,1490.0,1490.0,1.0,Precio x Unidad
4,8,1.0,1.0,7.798118e+12,1.0,TRAPO PISO LEYENDA GRIS.PREM.FRIZ. 50x60,1.0,uni.,10 y 11 S.A,2390.0,2390.0,1.0,Precio x Unidad


In [34]:
# productos con más de un precio distinto
precios_por_producto = filtered_df.groupby("id_producto")["productos_precio_lista"].nunique()
con_variacion = precios_por_producto[precios_por_producto > 1].sort_values(ascending=False)

print(f"Productos con precios distintos: {len(con_variacion)} de {precios_por_producto.shape[0]}")
display(con_variacion.head(10))

# ver detalle del que más variación tiene
id_ejemplo = con_variacion.index[0]
detalle = filtered_df[filtered_df["id_producto"] == id_ejemplo][["id_sucursal", "productos_descripcion", "productos_precio_lista"]]
print(f"\nDetalle id_producto={id_ejemplo:.0f}:")
display(detalle)

Productos con precios distintos: 26 de 4836


id_producto
2.003560e+12    2
2.203560e+12    2
2.403560e+12    2
7.450077e+12    2
7.622300e+12    2
7.790503e+12    2
7.791600e+12    2
7.791813e+12    2
7.791813e+12    2
7.791814e+12    2
Name: productos_precio_lista, dtype: int64


Detalle id_producto=2003560000000:


,id_sucursal,productos_descripcion,productos_precio_lista
1141,1.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
4540,2.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
8041,3.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
10520,4.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
14175,5.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
18407,6.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
22725,7.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
27433,8.0,QUESO CREMOSO SABORLAC EL KG.,8390.0
32095,9.0,QUESO CREMOSO SABORLAC EL KG.,8990.0
36718,10.0,QUESO CREMOSO SABORLAC EL KG.,8990.0


In [ ]:
# Cargar productos.csv de todos los ZIPs internos en un solo DataFrame
import pandas as pd
import zipfile
from pathlib import Path

COLS = [
    "id_comercio", "id_bandera", "id_sucursal", "id_producto",
    "productos_descripcion", "productos_marca",
    "productos_precio_lista",
]

frames = []
for zip_path in inner_zips:
    comercio_id = zip_path.stem.split("_")[2]  # ej: "comercio-sepa-8"
    with zipfile.ZipFile(zip_path) as z:
        if "productos.csv" not in z.namelist():
            continue
        with z.open("productos.csv") as f:
            df = pd.read_csv(f, sep="|", usecols=COLS, low_memory=False, encoding_errors="replace")
            df["source_zip"] = comercio_id
            frames.append(df)

all_df = pd.concat(frames, ignore_index=True)
all_df = all_df.dropna(subset=["id_producto", "productos_precio_lista"])
all_df["id_producto"] = all_df["id_producto"].astype("int64")

print(f"Filas totales:       {len(all_df):,}")
print(f"ZIPs cargados:       {len(frames)}")
print(f"Comercios únicos:    {all_df['id_comercio'].nunique()}")
print(f"Productos únicos:    {all_df['id_producto'].nunique()}")

In [ ]:
# Productos compartidos entre comercios y variación de precio entre ellos
productos_por_comercio = all_df.groupby("id_producto")["id_comercio"].nunique()
productos_multi_comercio = productos_por_comercio[productos_por_comercio > 1]

print(f"Productos en más de 1 comercio: {len(productos_multi_comercio):,} de {len(productos_por_comercio):,}")

# Para esos productos, calcular precio min/max por id_producto
variacion = (
    all_df[all_df["id_producto"].isin(productos_multi_comercio.index)]
    .groupby("id_producto")["productos_precio_lista"]
    .agg(precio_min="min", precio_max="max", n_comercios="count")
)
variacion["diferencia_pct"] = (variacion["precio_max"] - variacion["precio_min"]) / variacion["precio_min"] * 100
variacion = variacion[variacion["diferencia_pct"] > 0].sort_values("diferencia_pct", ascending=False)

print(f"Con diferencia de precio entre comercios: {len(variacion):,}")
display(variacion.head(15))

In [ ]:
# Detalle del producto con mayor diferencia porcentual entre comercios
id_ejemplo = variacion.index[0]
detalle = (
    all_df[all_df["id_producto"] == id_ejemplo]
    [["id_comercio", "productos_descripcion", "productos_precio_lista"]]
    .drop_duplicates()
    .sort_values("productos_precio_lista")
)
print(f"id_producto: {id_ejemplo}")
display(detalle)